In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score

In [2]:
import pandas as pd
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [3]:
def fillna(df):
    df = df.copy()  # 명시적 복사
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    df['Cabin'] = df['Cabin'].fillna('N')
    df['Embarked'] = df['Embarked'].fillna('N')
    df['Fare'] = df['Fare'].fillna(0)
    return df

def drop_features(df):
    return df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

def format_features(df):
    df = df.copy()
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        df[feature] = le.fit_transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [4]:
def get_clf_eval(y_test, pred):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)

    print(confusion)
    print('*'*20)
    print(accuracy, precision, recall)

In [ ]:
titanic_df = transform_features(df)
y_titanic_df = titanic_df['Survived'] # 피쳐와 타겟분리 
X_titanic_df = titanic_df.drop('Survived', axis=1) # 훈련과 테스트 데이터 분류
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df,
                                                    y_titanic_df,
                                                    test_size=0.2,
                                                    random_state=0 )

# 성능비교- 로지스틱 회귀

In [6]:
#로지스틱회귀 분류모델 생성
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=2000)
lr_clf.fit(X_train, y_train)
pred = lr_clf.predict(X_test)

#정확도, 정밀도, 재현율
get_clf_eval(y_test, pred)

[[92 18]
 [16 53]]
********************
0.8100558659217877 0.7464788732394366 0.7681159420289855


# 단순가설 분류길를 이용한 성능 기준 확인
남성 > 사망, 여성 > 생존

In [ ]:
from sklearn.base import BaseEstimator
import numpy as np
class MyDummyClassifier(BaseEstimator):
  def fit(self, X, y):
    pass

  def predict(self, X):
    pred = np.zeros((X.shape[0],1))
    for i in range(X.shape[0]): # 테스트 데이터 갯수만큼
      if X['Sex'].iloc[i] == 1:
        pred[i]=0
      else :
        pred[i]=1
    return pred

In [8]:
my_clf = MyDummyClassifier()
my_clf.fit(X_train, y_train)
pred = my_clf.predict(X_test)
accuracy = accuracy_score(y_test, pred)
print(accuracy)

0.7877094972067039


# 랜덤 포레스트, knn의 정확도, 정밀도 ,재현율 비교하기 

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd

# 1. 모델 초기화
# Random Forest: 트리 100개 사용
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# KNN: 가장 가까운 이웃 5개 기준 (필요시 n_neighbors 조절 가능)
knn_model = KNeighborsClassifier(n_neighbors=5)

# 2. 모델 학습
rf_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)

# 3. 예측 수행
rf_pred = rf_model.predict(X_test)
knn_pred = knn_model.predict(X_test)

# 4. 성능 지표 계산 및 비교
def get_metrics(y_test, y_pred, model_name):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    return {
        'Model': model_name,
        'Accuracy': f"{accuracy:.4f}",
        'Precision': f"{precision:.4f}",
        'Recall': f"{recall:.4f}"
    }

rf_results = get_metrics(y_test, rf_pred, 'Random Forest')
knn_results = get_metrics(y_test, knn_pred, 'KNN')

# 결과 비교 테이블 출력
comparison_df = pd.DataFrame([rf_results, knn_results])
print("=== 타이타닉 생존 예측 모델 성능 비교 ===")
print(comparison_df)

=== 타이타닉 생존 예측 모델 성능 비교 ===
           Model Accuracy Precision  Recall
0  Random Forest   0.8380    0.8448  0.7101
1            KNN   0.7374    0.7037  0.5507


In [10]:
from sklearn.ensemble import VotingClassifier

# 1. 앙상블 모델 설정 (Soft Voting 방식)
# Soft Voting: 각 모델의 확률값을 평균 내어 결정 (일반적으로 성능이 더 좋음)
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('knn', knn_model)],
    voting='soft'
)

# 2. 앙상블 모델 학습
ensemble_model.fit(X_train, y_train)

# 3. 예측 및 성능 계산
ensemble_pred = ensemble_model.predict(X_test)
ensemble_results = get_metrics(y_test, ensemble_pred, 'Ensemble (RF+KNN)')

# 4. 기존 결과와 합쳐서 비교
final_comparison = pd.DataFrame([rf_results, knn_results, ensemble_results])
print("=== 앙상블 포함 성능 비교 ===")
print(final_comparison)

=== 앙상블 포함 성능 비교 ===
               Model Accuracy Precision  Recall
0      Random Forest   0.8380    0.8448  0.7101
1                KNN   0.7374    0.7037  0.5507
2  Ensemble (RF+KNN)   0.8101    0.7869  0.6957


# 분류모델의 임계치의 확인

In [14]:
pred_proba = lr_clf.predict_proba(X_test)
pos_proba=pred_proba[:,1] # 양성

threshold = 0.4
custom_proba=(pos_proba >= threshold).astype(int)
confusion = confusion_matrix(y_test, custom_proba)
confusion

array([[86, 24],
       [13, 56]])

In [15]:
get_clf_eval(y_test, custom_proba)

[[86 24]
 [13 56]]
********************
0.7932960893854749 0.7 0.8115942028985508
